<a href="https://colab.research.google.com/github/michael-sorani/LM4GeoAI/blob/main/Exercises/EX_2_Text_to_Geo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ex 2: **Text 2 Geo Data**
### created by Etzion Harari | Geo-AI Course

[**https://github.com/EtzionR/LM4GeoAI**](https://github.com/EtzionR/LM4GeoAI)

## Imports

In [1]:
from transformers import pipeline, AutoModelForTokenClassification, AutoTokenizer
from geopy.geocoders import Nominatim
from time import sleep as wait
from tqdm import tqdm

import networkx as nx
import pandas as pd
import folium

## Clone git Repo
[https://github.com/EtzionR/LM4GeoAI](https://github.com/EtzionR/LM4GeoAI)

In [2]:
%%bash
rm -rf LM4GeoAI
git clone https://github.com/EtzionR/LM4GeoAI.git

Cloning into 'LM4GeoAI'...


## Load Data

In [3]:
df = pd.read_csv('LM4GeoAI/Data/data.csv')

print(f'Dataframe shape: {df.shape}')

df.head()

Dataframe shape: (23072, 2)


,Text,Source
0,"Last week, Sen. Malcolm Wallop -LRB- R., Wyo. ...",ontonotes5
1,Rules that set standards for products or gover...,ontonotes5
2,Determining when handicapped access is require...,ontonotes5
3,"``It's very costly and time-consuming ,'' says...",ontonotes5
4,"Next to medical insurance, ``costs of complian...",ontonotes5


# --------------------------------------------------------------------

## Q1

#### A) Create a Hugging Face pipeline for NER with average aggregation_strategy enabled. Please use [dslim/bert-base-NER](https://huggingface.co/dslim/bert-base-NER) model.

#### B) Apply the pipeline to a sample sentence containing people, organizations, and locations.

#### C) Display the NER output in a pandas DataFrame.

In [4]:
model_name = "dslim/bert-base-NER"
# Create the NER pipeline with aggregation_strategy='average'
classifier = pipeline("ner", model=model_name, aggregation_strategy="average")

# Sample sentence
sample_sentence = "Apple Inc. was founded by Steve Jobs and Steve Wozniak in Cupertino, California. It has offices in London and New York."

# Apply the pipeline to the sample sentence
ner_output = classifier(sample_sentence)

# Display the NER output in a pandas DataFrame
ner_df = pd.DataFrame(ner_output)
display(ner_df)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,entity_group,score,word,start,end
0,ORG,0.999409,Apple Inc,0,9
1,PER,0.995844,Steve Jobs,26,36
2,PER,0.995397,Steve Wozniak,41,54
3,LOC,0.665649,Cupertino,58,67
4,LOC,0.999561,California,69,79
5,LOC,0.999295,London,99,105
6,LOC,0.999109,New York,110,118


## Q2

#### A) Select the 10% first rows from the DataFrame.

#### B) Apply the NER model on each of the selected texts.

#### C) For every detected entity, attach metadata such as the **original text** and its **source**.

#### D) Aggregate all entity-level outputs into a single pandas DataFrame and display its shape.

In [5]:
df_sample = df.head(round(0.1 * len(df))).copy()

df_sample['ner'] = df_sample['Text'].apply(lambda x: classifier(x))

df_expanded_ner = df_sample.explode('ner')
entity_details = pd.json_normalize(df_expanded_ner['ner'])
entity_df = pd.concat([
    df_expanded_ner[['Text', 'Source']].reset_index(drop=True),
    entity_details.reset_index(drop=True)
], axis=1)
entity_df = entity_df.rename(columns={'Text': 'original_text'})

print(f'Shape of the aggregated entity DataFrame: {entity_df.shape}')
entity_df.head()

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Shape of the aggregated entity DataFrame: (6658, 7)


,original_text,Source,entity_group,score,word,start,end
0,"Last week, Sen. Malcolm Wallop -LRB- R., Wyo. ...",ontonotes5,PER,0.999573,Malcolm Wallop,16.0,30.0
1,"Last week, Sen. Malcolm Wallop -LRB- R., Wyo. ...",ontonotes5,ORG,0.500653,LRB,32.0,35.0
2,"Last week, Sen. Malcolm Wallop -LRB- R., Wyo. ...",ontonotes5,LOC,0.879631,R,37.0,38.0
3,"Last week, Sen. Malcolm Wallop -LRB- R., Wyo. ...",ontonotes5,LOC,0.428096,Wyo,41.0,44.0
4,"Last week, Sen. Malcolm Wallop -LRB- R., Wyo. ...",ontonotes5,ORG,0.499855,RRB,47.0,50.0


## Q3

#### 1) Filter the NER output DataFrame to include only location entities.

#### 2) Count the total number of location entity occurrences.

#### 3) Compute the top-10 most frequently occurring location names.

#### 4) Calculate the total number of mentions among the top-K entities and their percentage relative to all location entities.

#### 5) Display the resulting summary table.

In [11]:
location_df = entity_df[entity_df['entity_group'] == 'LOC']
print(f'Total number of location entity occurrences: {len(location_df)}')

top_10_locations = pd.DataFrame(location_df.groupby('word').size().sort_values(ascending=False).head(10))
top_10_locations.reset_index(inplace=True)
top_10_locations.columns = ['location', 'count']
top_10_locations['percentage'] = (top_10_locations['count'] / len(location_df) * 100).round(2)
print('Top 10 most frequently occurring location:')
display(top_10_locations)

Total number of location entity occurrences: 3316
Top 10 most frequently occurring location:


,location,count,percentage
0,U. S.,387,11.67
1,New York,154,4.64
2,Japan,69,2.08
3,California,64,1.93
4,China,55,1.66
5,London,45,1.36
6,Poland,41,1.24
7,Britain,41,1.24
8,Washington,40,1.21
9,Los Angeles,38,1.15


## Q4

#### 1) Initialize a Nominatim geocoder with a custom user agent and timeout.

#### 2) Geocode the example place name into a geographic location.

#### 3) Display the output geocoding result, include the X (longtitue) and Y (latitude) coordinates

In [12]:
# 1) Initialize a Nominatim geocoder with a custom user agent and timeout.
geolocator = Nominatim(user_agent="geo_ai_course_app", timeout=10)

# 2) Geocode an example place name into a geographic location.
example_place = "Paris, France"
location = geolocator.geocode(example_place)

# 3) Display the output geocoding result, include the X (longitude) and Y (latitude) coordinates
if location:
    print(f"Geocoding result for '{example_place}':")
    print(f"Address: {location.address}")
    print(f"Latitude (Y): {location.latitude}")
    print(f"Longitude (X): {location.longitude}")
else:
    print(f"Could not geocode '{example_place}'")

Geocoding result for 'Paris, France':
Address: Paris, Île-de-France, France métropolitaine, France
Latitude (Y): 48.8534951
Longitude (X): 2.3483915


## Q5

#### Geocode each place name in the top 10 place name dataframe. Add to each entry in the dataframe its coordinates. Please define a delay interval of 1. seconds (at least) between each call to Nominatim.


In [17]:
def geocode_location(location_name):
    try:
        wait(1)
        location = geolocator.geocode(location_name)
        if location:
            return pd.Series({
                'address': location.address,
                'longitude': location.longitude,
                'latitude': location.latitude
            })
        else:
            # Return NaNs for location not found
            return pd.Series({
                'address': None,
                'longitude': None,
                'latitude': None
            })
    except Exception as e:
        print(f"Error geocoding '{location_name}': {e}")
        return pd.Series({
            'address': None,
            'longitude': None,
            'latitude': None
        })


top_10_locations[['address', 'longitude', 'latitude']] = top_10_locations['location'].apply(geocode_location)
top_10_locations

,location,count,percentage,address,longitude,latitude
0,U. S.,387,11.67,United States,-100.445882,39.783730
1,New York,154,4.64,"New York, United States",-74.006015,40.712728
2,Japan,69,2.08,日本,139.239418,36.574844
3,California,64,1.93,"California, United States",-118.755997,36.701463
4,China,55,1.66,中国,108.923707,34.541225
5,London,45,1.36,"Greater London, England, United Kingdom",-0.127765,51.507446
6,Poland,41,1.24,Polska,19.134422,52.215933
7,Britain,41,1.24,"Great Britain, United Kingdom",-1.918153,54.315159
8,Washington,40,1.21,"Washington, District of Columbia, United States",-77.036385,38.895098
9,Los Angeles,38,1.15,"Los Angeles, Los Angeles County, California, U...",-118.242766,34.053691


## Q6

#### Create folium map of the Top 10 places in the Corpus

In [18]:
locations_with_coords = top_10_locations.dropna(subset=['latitude', 'longitude'])

if not locations_with_coords.empty:
    map_center = [locations_with_coords['latitude'].mean(), locations_with_coords['longitude'].mean()]
else:
    map_center = [0, 0] # Default center if no valid coordinates

m = folium.Map(location=map_center, zoom_start=2) # Adjust zoom as needed

# Add markers for each of the top 10 locations
for index, row in locations_with_coords.iterrows():
    # Ensure latitude and longitude are not None before adding marker
    if pd.notna(row['latitude']) and pd.notna(row['longitude']):
        popup_text = (
            f"<b>Location:</b> {row['location']}<br>"
            f"<b>Count:</b> {row['count']}<br>"
            f"<b>Percentage:</b> {row['percentage']}%"
        )
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=popup_text,
            tooltip=row['location']
        ).add_to(m)

# Display the map
display(m)

## Q7

#### Construct an undirected graph where nodes represent the top-10 location entities and edges indicate that two locations co-occur in the **same text**. Track edge weights based on the number of co-occurrences and report the final number of nodes and edges.

In [19]:
# 1) Get the list of top 10 locations to consider for graph nodes
top_10_location_names = top_10_locations['location'].tolist()

# Filter the entity_df to include only location entities and the top 10 relevant ones
filtered_locations = entity_df[entity_df['entity_group'] == 'LOC']
filtered_locations = filtered_locations[filtered_locations['word'].isin(top_10_location_names)]

# Group by original_text to find co-occurrences within each text
co_occurrence_groups = filtered_locations.groupby('original_text')['word'].apply(list)

# Initialize the graph
G = nx.Graph()

# Add nodes for each of the top 10 locations
for loc in top_10_location_names:
    G.add_node(loc)

# Iterate through co-occurrence groups to add edges
for locations_in_text in co_occurrence_groups:
    # Ensure unique locations within a text to avoid self-loops or duplicate counts from same text
    unique_locations_in_text = list(set(locations_in_text))

    # Add edges for all pairs of locations in the same text
    for i in range(len(unique_locations_in_text)):
        for j in range(i + 1, len(unique_locations_in_text)):
            loc1 = unique_locations_in_text[i]
            loc2 = unique_locations_in_text[j]

            # Only add edge if both locations are in our top 10 list
            if loc1 in top_10_location_names and loc2 in top_10_location_names:
                if G.has_edge(loc1, loc2):
                    G[loc1][loc2]['weight'] += 1
                else:
                    G.add_edge(loc1, loc2, weight=1)

# Report the final number of nodes and edges
print(f"Number of nodes in the graph: {G.number_of_nodes()}")
print(f"Number of edges in the graph: {G.number_of_edges()}")

Number of nodes in the graph: 10
Number of edges in the graph: 15


## Q8

#### Display the constructed graph with folium. Draw each edge in the graph as polyline between the two places it connect.

In [20]:
# Re-initialize the Folium map for the graph visualization
# Filter out rows where latitude or longitude might be None
locations_with_coords = top_10_locations.dropna(subset=['latitude', 'longitude'])

# Calculate the mean latitude and longitude for centering the map
if not locations_with_coords.empty:
    map_center = [locations_with_coords['latitude'].mean(), locations_with_coords['longitude'].mean()]
else:
    map_center = [0, 0] # Default center if no valid coordinates

m_graph = folium.Map(location=map_center, zoom_start=2) # Adjust zoom as needed

# Add markers for each of the top 10 locations (nodes)
for index, row in locations_with_coords.iterrows():
    if pd.notna(row['latitude']) and pd.notna(row['longitude']):
        popup_text = (
            f"<b>Location:</b> {row['location']}<br>"
            f"<b>Count:</b> {row['count']}<br>"
            f"<b>Percentage:</b> {row['percentage']}%"
        )
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=popup_text,
            tooltip=row['location'],
            icon=folium.Icon(color='blue') # Differentiate from previous map markers if any
        ).add_to(m_graph)

# Add edges (polylines) to the map
for u, v, data in G.edges(data=True):
    loc1_coords = locations_with_coords[locations_with_coords['location'] == u][['latitude', 'longitude']].values
    loc2_coords = locations_with_coords[locations_with_coords['location'] == v][['latitude', 'longitude']].values

    if len(loc1_coords) > 0 and len(loc2_coords) > 0:
        loc1_lat, loc1_lon = loc1_coords[0]
        loc2_lat, loc2_lon = loc2_coords[0]

        # Create a polyline between the two locations
        # Line weight can be based on the edge weight
        line_weight = data['weight'] / 2.0 + 1 # Scale weight for visibility
        folium.PolyLine(
            locations=[[loc1_lat, loc1_lon], [loc2_lat, loc2_lon]],
            color='red',
            weight=line_weight,
            tooltip=f"Co-occurrence: {u} - {v} (Weight: {data['weight']})"
        ).add_to(m_graph)

# Display the map
display(m_graph)

## Q - Bonus

#### 1) Identify the top three Person (PER) entities based on their frequency of occurrence in the corpus.

#### 2) Select from the output entities dataframe only the texts that mention at least one of these top three PER entities.

#### 3) For each Location (LOC) entity appearing in the selected texts, retrieve its geographic coordinates using the Nominatim geocoding service.

#### 4) Visualize the results on a Folium map, ensuring that all locations co-occurring with the same person are displayed using the same color.

In [21]:
# 1) Identify the top three Person (PER) entities
person_entities = entity_df[entity_df['entity_group'] == 'PER']
top_3_persons = person_entities['word'].value_counts().head(3).index.tolist()
print(f"Top 3 Person entities: {top_3_persons}")

# 2) Select from the output entities DataFrame only the texts that mention at least one of these top three PER entities.
texts_with_top_persons = entity_df[entity_df['word'].isin(top_3_persons)]['original_text'].unique()

# Filter entity_df for locations only in these texts
locations_co_occurring_with_top_persons = entity_df[
    (entity_df['entity_group'] == 'LOC') &
    (entity_df['original_text'].isin(texts_with_top_persons))
].copy()

# Group locations by the person they co-occur with in the original text
# This requires re-processing the original texts to associate locations with specific persons

# Create a mapping of text_id to a list of persons in that text
person_in_text_map = {}
for text in texts_with_top_persons:
    persons_found = entity_df[
        (entity_df['original_text'] == text) &
        (entity_df['entity_group'] == 'PER') &
        (entity_df['word'].isin(top_3_persons))
    ]['word'].tolist()
    if persons_found:
        person_in_text_map[text] = list(set(persons_found))

# Prepare data for geocoding and mapping
locations_to_map = []
for text_id, persons_list in person_in_text_map.items():
    locations_in_text = entity_df[
        (entity_df['original_text'] == text_id) &
        (entity_df['entity_group'] == 'LOC')
    ]['word'].unique()
    for loc in locations_in_text:
        for person in persons_list:
            locations_to_map.append({'location': loc, 'person': person, 'original_text': text_id})

locations_to_map_df = pd.DataFrame(locations_to_map).drop_duplicates(subset=['location', 'person'])

# 3) For each Location (LOC) entity appearing in the selected texts, retrieve its geographic coordinates
# Use the already defined geocode_location function from Q5
# Add a 'unique_id' for easier tracking if needed, though 'location' and 'person' combo is unique here

# Create a unique list of locations to geocode to avoid redundant API calls
unique_locations_to_geocode = locations_to_map_df['location'].unique()
geocoded_results = {}
for loc_name in tqdm(unique_locations_to_geocode, desc="Geocoding bonus locations"):
    result = geocode_location(loc_name)
    geocoded_results[loc_name] = result.to_dict()

# Merge geocoded results back to locations_to_map_df
geocoded_df = pd.DataFrame.from_dict(geocoded_results, orient='index')
geocoded_df.index.name = 'location'
geocoded_df.reset_index(inplace=True)

locations_final_df = pd.merge(locations_to_map_df, geocoded_df, on='location', how='left')
locations_final_df.dropna(subset=['latitude', 'longitude'], inplace=True)

# Define colors for each top person
colors = ['red', 'green', 'purple', 'orange', 'darkred', 'lightred', 'blue', 'darkgreen', 'cadetblue', 'darkpurple'] # More colors than needed
person_color_map = {person: colors[i] for i, person in enumerate(top_3_persons)}

# 4) Visualize the results on a Folium map
m_bonus = folium.Map(location=[locations_final_df['latitude'].mean(), locations_final_df['longitude'].mean()], zoom_start=2)

for index, row in locations_final_df.iterrows():
    person = row['person']
    color = person_color_map.get(person, 'gray') # Default to gray if person not in top 3 (shouldn't happen here)

    popup_text = (
        f"<b>Location:</b> {row['location']}<br>"
        f"<b>Person:</b> {person}"
    )
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=popup_text,
        tooltip=f"{row['location']} (with {person})",
        icon=folium.Icon(color=color)
    ).add_to(m_bonus)

display(m_bonus)

Top 3 Person entities: ['Bush', 'Hugo', 'Thatcher']


Geocoding bonus locations: 100%|██████████| 39/39 [00:53<00:00,  1.37s/it]


Create by Etzion Harari | Geo-AI Course | [https://github.com/EtzionR/LM4GeoAI](https://github.com/EtzionR/LM4GeoAI)